Для інсталювання необхідного пакету в середовищі Anaconda потрібно ввести в терміналі Anaconda:

```python
pip install pulp
```

Для інсталювання необхідного пакету в середовищі Google Colab потрібно ввести першій комірці:

```python
!pip install pulp
```

Більше деталей за [посиланням](https://colab.research.google.com/notebooks/snippets/importing_libraries.ipynb).

## Дано

Нехай маємо математичну модель траспортної задачі лінійного програмування:

\begin{align}
max \hspace{1cm} f & = 90x_1 + 110x_2 + 150x_3, \\
s.t. \hspace{0.5cm} x_1 + 2x_2 + 4x_3 \leq 360; \\
\hspace{0.5cm} 2x_1 + 4x_2 + 2x_3 \leq 520; \\
\hspace{0.5cm} x_1 + x_2 + 2x_3 \leq 220; \\
x_1 & \geq 0; \\
x_2 & \geq 0; \\
x_3 & \geq 0; \\
\end{align}

## Блок 1. Ініціалізація вхідних даних

In [134]:
from pulp import *

In [135]:
# Склади та торгівельні точки: List
Warehouse = ['A1', 'A2']
Branch = ['B1', 'B2', 'B3', 'B4']

In [136]:
# Об'єми вивозу
demand = {'A1' : 60, 'A2' : 40}

In [137]:
# Об'єми поставок
supply = {'B1' : 20, 'B2' : 30, 'B3' : 45, 'B4' : 15}

In [138]:
# Витрати на перевезення одиниці вантажу із складів до торгівельних точок
cost = {'A1': {'B1' : 3, 'B2' : 4, 'B3' : 6, 'B4' : 7},
        'A2': {'B1' : 2, 'B2' : 5, 'B3' : 8, 'B4' : 1},
       }

## Блок 2. Реалізація транспортної задачі лінійного програмування

In [139]:
# Створення об'єкту математичної моделі
probTrans = LpProblem("ТЗЛП", LpMinimize)

In [140]:
# Оголошення матриці, що моделює всі можливі шляхи перевезення 
routes = [(i,j) for i in Warehouse for j in Branch]
print(routes)

[('A1', 'B1'), ('A1', 'B2'), ('A1', 'B3'), ('A1', 'B4'), ('A2', 'B1'), ('A2', 'B2'), ('A2', 'B3'), ('A2', 'B4')]


In [141]:
# Оголошення змінних мат. моделі
amount_vars = LpVariable.dicts(name="X", indexs=(Warehouse,Branch), lowBound=0, upBound=None)
print(amount_vars)

{'A1': {'B1': X_A1_B1, 'B2': X_A1_B2, 'B3': X_A1_B3, 'B4': X_A1_B4}, 'A2': {'B1': X_A2_B1, 'B2': X_A2_B2, 'B3': X_A2_B3, 'B4': X_A2_B4}}


C:\Users\RyzenPC\AppData\Local\Temp\ipykernel_14640\2588722137.py:2: DeprecationWarning: 'indexs' is deprecated; use 'indices'.
  amount_vars = LpVariable.dicts(name="X", indexs=(Warehouse,Branch), lowBound=0, upBound=None)


In [142]:
# Оголошення та ініціалізація цільової функції
# lpSum - обчислення суми добутків C*X
probTrans += lpSum(amount_vars[i][j]*cost[i][j] for (i,j) in routes)

In [143]:
# Оголошення та ініціалізація обмежень моделі
for j in Branch:
    probTrans += lpSum(amount_vars[i][j] for i in Warehouse) <= supply[j]

for i in Warehouse:
    probTrans += lpSum(amount_vars[i][j] for j in Branch) == demand[i]

In [144]:
# Відображення математичної моделі
print(probTrans)

ТЗЛП:
MINIMIZE
3*X_A1_B1 + 4*X_A1_B2 + 6*X_A1_B3 + 7*X_A1_B4 + 2*X_A2_B1 + 5*X_A2_B2 + 8*X_A2_B3 + 1*X_A2_B4 + 0
SUBJECT TO
_C1: X_A1_B1 + X_A2_B1 <= 20

_C2: X_A1_B2 + X_A2_B2 <= 30

_C3: X_A1_B3 + X_A2_B3 <= 45

_C4: X_A1_B4 + X_A2_B4 <= 15

_C5: X_A1_B1 + X_A1_B2 + X_A1_B3 + X_A1_B4 = 60

_C6: X_A2_B1 + X_A2_B2 + X_A2_B3 + X_A2_B4 = 40

VARIABLES
X_A1_B1 Continuous
X_A1_B2 Continuous
X_A1_B3 Continuous
X_A1_B4 Continuous
X_A2_B1 Continuous
X_A2_B2 Continuous
X_A2_B3 Continuous
X_A2_B4 Continuous



In [145]:
# Розв'язок ЗЛП
probTrans.solve(PULP_CBC_CMD(msg=1))

1

## Блок 3. Виведення результату розв'язку транспортної задачі лінійного програмування

In [146]:
# Виведення результату оптимізації
print("Status:", LpStatus[probTrans.status])

Status: Optimal


In [147]:
# Виведення значень перевезень
for v in probTrans.variables():
    if v.varValue > 0:
        print(v.name, "=", v.varValue)

X_A1_B2 = 25.0
X_A1_B3 = 35.0
X_A2_B1 = 20.0
X_A2_B2 = 5.0
X_A2_B4 = 15.0


In [148]:
print("Оптимальне значення цільової функції: z =", value(probTrans.objective))

Оптимальне значення цільової функції: z = 390.0
